# 12. VAAS v2 Model Variants Comparison

This notebook compares the three released **VAAS v2** model variants on the same images.

It covers:

- loading `v2-base-df2023`, `v2-medium-df2023`, and `v2-large-df2023`
- running inference on the same inputs
- comparing `S_F`, `S_P`, and `S_H`
- generating visual outputs for qualitative comparison
- ranking model responses by anomaly score

## 1. Install dependencies

In [ ]:
!pip install -q vaas torch torchvision

## 2. Imports

In [ ]:
from io import BytesIO

import requests
import pandas as pd
from PIL import Image
from IPython.display import Image as IPImage, display

from vaas.inference.pipeline import VAASPipeline

## 3. Load example images

This notebook uses two public examples from the VAAS repository so that the same comparison can be reproduced easily.

In [ ]:
image_urls = {
    "example_1": "https://raw.githubusercontent.com/OBA-Research/VAAS/main/examples/images/COCO_DF_C110B00000_00539519.jpg",
    "example_2": "https://raw.githubusercontent.com/OBA-Research/VAAS/main/examples/images/COCO_DF_S000B00000_00120651.jpg",
}

images = {
    name: Image.open(BytesIO(requests.get(url).content)).convert("RGB")
    for name, url in image_urls.items()
}

images["example_1"]

## 4. Load VAAS v2 variants

We load the three public model variants released on Hugging Face.

In [ ]:
variants = [
    "v2-base-df2023",
    "v2-medium-df2023",
    "v2-large-df2023",
]

pipelines = {
    variant: VAASPipeline.from_pretrained(
        repo_id="OBA-Research/vaas",
        model_variant=variant,
        device="cpu",   # change to "cuda" if GPU is available
        alpha=0.5,
    )
    for variant in variants
}

list(pipelines.keys())

## 5. Run inference across variants

For each image, we compare how the three variants score the same input.

In [ ]:
rows = []

for image_name, image in images.items():
    for variant, pipeline in pipelines.items():
        out = pipeline(image)
        rows.append(
            {
                "image": image_name,
                "variant": variant,
                "S_F": out["S_F"],
                "S_P": out["S_P"],
                "S_H": out["S_H"],
            }
        )

df = pd.DataFrame(rows)
df

## 6. Compare hybrid anomaly scores

Sorting by `S_H` makes it easier to inspect how each model responds to the same image.

In [ ]:
df_sorted = df.sort_values(["image", "S_H"], ascending=[True, False]).reset_index(drop=True)
df_sorted

## 7. Quick per-image comparison

In [ ]:
for image_name in df["image"].unique():
    print(f"\n=== {image_name} ===")
    subset = df[df["image"] == image_name][["variant", "S_F", "S_P", "S_H"]]
    print(subset.sort_values("S_H", ascending=False).to_string(index=False))

## 8. Generate qualitative visual outputs

This helps compare how each released model variant localises the same anomaly.

In [ ]:
target_image_name = "example_1"
target_image = images[target_image_name]

for variant, pipeline in pipelines.items():
    save_path = f"{variant}_{target_image_name}.png"
    pipeline.visualize(
        image=target_image,
        save_path=save_path,
        mode="all",
        threshold=0.5,
    )
    print(f"Saved: {save_path}")

## 9. Display the visual outputs

In [ ]:
for variant in variants:
    display(IPImage(f"{variant}_{target_image_name}.png"))

## 10. Interpret the comparison

Typical use:

- `v2-base-df2023` for fast lightweight testing
- `v2-medium-df2023` for balanced use
- `v2-large-df2023` for strongest released variant

## 11. Optional: select the strongest responding variant per image

In [ ]:
best_per_image = (
    df.sort_values("S_H", ascending=False)
      .groupby("image", as_index=False)
      .first()[["image", "variant", "S_H"]]
)

best_per_image

## 12. Summary

In this notebook you:

- loaded the three released VAAS v2 variants
- ran them on the same example images
- compared anomaly scores
- generated qualitative visual outputs
- identified the strongest responding model variant per image

Next notebook: [**13_forward_detailed_and_metadata.ipynb**](https://colab.research.google.com/drive/1X0rttevFiJJDMjuJL_AVZeystCzliWQg?usp=sharing)